# Költségtérítési elemzés

Ez a jegyzetfüzet bemutatja, hogyan hozhatók létre olyan ügynökök, amelyek pluginok segítségével dolgozzák fel az utazási költségeket helyi blokk képekből, készítenek költségtérítési e-mailt, és megjelenítik a költségadatokat kördiagramon. Az ügynökök dinamikusan választanak funkciókat a feladat kontextusa alapján.

Lépések:
1. Az OCR ügynök feldolgozza a helyi blokk képet és kinyeri az utazási költségadatokat.
2. Az E-mail ügynök létrehozza a költségtérítési e-mailt.

### Az utazási költséghelyzet példája:
Képzeld el, hogy alkalmazottként üzleti tárgyalásra utazol egy másik városba. A céged szabályzata szerint minden ésszerű utazási költséget megtérítenek. Íme egy bontás a lehetséges utazási költségekről:
- Közlekedés:
Retúr repülőjegy az otthoni városod és a célváros között.
Taxi vagy utazásmegosztó szolgáltatások az repülőtérre és vissza.
Helyi közlekedés a célvárosban (például tömegközlekedés, autóbérlés vagy taxik).

- Szállás:
Három éjszakás szállás egy középkategóriás üzleti hotelben, a tárgyalás helyszínéhez közel.

- Ételek:
Napi étkezési keret reggelire, ebédre és vacsorára, a cég napidíjas szabályzata alapján.

- Egyéb költségek:
Parkolási díjak a repülőtéren.
Internet-hozzáférési díjak a hotelben.
Borravalók vagy apró kiszolgálási díjak.

- Dokumentáció:
Minden blokkot benyújtasz (repülőjegyek, taxik, hotel, ételek stb.) és egy kitöltött költségelszámolási jelentést a térítéshez.


## Szükséges könyvtárak importálása

Importálja a jegyzetfüzethez szükséges könyvtárakat és modulokat.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Költségmodellek definiálása

 Hozzon létre egy Pydantic modellt az egyéni költségekhez, és egy ExpenseFormatter osztályt, amely a felhasználói lekérdezést strukturált költségadatokká alakítja.

 Minden költséget a következő formátumban képviselnek:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Eszközök meghatározása - E-mail generálása

Hozzon létre egy eszközfüggvényt egy költségtérítési igény benyújtásához szükséges e-mail generálásához.
- Ez az eszköz a Microsoft Agent Framework `@tool` dekorátorát használja.
- Kiszámítja a költségek teljes összegét, és az adatokat egy e-mail törzsbe formázza.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Eszköz az utazási költségek kinyerésére a blokk képekből

Hozzon létre egy eszközfüggvényt az utazási költségek kinyerésére a blokk képekből.
- Ez az eszköz a Microsoft Agent Framework `@tool` dekorátorát használja.
- Beolvassa a blokk képét, base64-kódolja, és visszaadja az adat-URI-t az ügynök elemzéséhez.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Költségek feldolgozása

Határozza meg az ügynököket, és kösse össze őket egy szekvenciális munkafolyamatba a `WorkflowBuilder` segítségével.
- Az OCR ügynök a `load_receipt_image` eszköz használatával kinyeri a strukturált költségadatokat a blokk képről.
- Az Email ügynök a kinyert adatok alapján professzionális költségigénylő e-mailt készít a `generate_expense_email` eszközzel.
- A `WorkflowBuilder` az `add_edge`-del egy szekvenciális csővezetéket hoz létre: OCR ügynök → Email ügynök.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Fő függvény

Építse fel a szekvenciális munkafolyamatot, és futtassa azt a nyugta kép feldolgozásához és a költségtérítési e-mail létrehozásához.


> **Megjegyzés:** Ez a munkafolyamat jelenleg a számla képét base64 szövegként adja át, amit a legtöbb csevegőmodell (beleértve a gpt-4o-t is) nem kezel képként.
> Ez meghaladhatja a modell kontextusablakának méretét is. Ajánlott OCR futtatása Azure AI Vision-nel (vagy más OCR eszközzel), és csak a kinyert szöveget átadni, vagy úgy átdolgozni, hogy a képet `image_url` üzenetként küldje el.
> Ha csak a kontextushibákat akarja elkerülni, próbáljon kisebb számlaképet vagy nagyobb kontextusablakú modellt használni.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
